# Data Quality Uniqueness Check Demo

This notebook demonstrates the data quality uniqueness checking framework.

## Purpose
Demonstrates uniqueness checks through two scenarios:
1. **Single column check** on employee data (id)
2. **Composite uniqueness checks** on multiple column combinations

## Scenarios

### Scenario 1: CSV Data (employees.csv)
* Single column: `id` should be unique
* Composite: `name + department` combination should be unique

### Scenario 2: Hard-Coded Contact Data
* Composite: `first_name + last_name + phone` combination should be unique
* Shows why composite keys matter (same phone can belong to different people)

## What it does
* Loads test data from CSV and hard-coded sources
* Executes both single-column and composite uniqueness checks
* Identifies duplicate records and shows actual duplicate rows
* Combines results from both scenarios
* Saves results to a Delta table for tracking and reporting

## Output
* Uniqueness report showing duplicate counts and percentages per column
* Results saved to `workspace.default.uniqueness_report` table

In [0]:
"""Scenario 1: Employee CSV Data
Checks:
1. Single column uniqueness: id
2. Composite uniqueness: name + department
"""

# Import required libraries
import sys
import yaml

# Add DQ checks module to path
sys.path.append('/Workspace/Repos/maha.b.lakshmi@gmail.com/data-quality-testing/src/checks')
from dq_checks import check_uniqueness, get_duplicate_rows, check_uniqueness_composite

# Load configuration
config_path = "/Workspace/Repos/maha.b.lakshmi@gmail.com/data-quality-testing/src/config/config.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

# Load employee test data from CSV
employees = spark.read.option("header", True).option("inferSchema", True) \
    .csv("/Workspace/Repos/maha.b.lakshmi@gmail.com/data-quality-testing/tests/test_data/employees.csv")

print("📊 Employee Data:")
employees.display()

# Check 1: Single column uniqueness - id
print("\n" + "="*60)
print("CHECK 1: ID Column Uniqueness")
print("="*60)
id_result = check_uniqueness(employees, "id")
print(f"Total Rows: {id_result['total_rows']}")
print(f"Distinct Count: {id_result['distinct_count']}")
print(f"Duplicate Count: {id_result['duplicate_count']}")
print(f"Passed: {'✅' if id_result['passed'] else '❌'}")

if id_result['duplicate_count'] > 0:
    print("\n⚠️ Duplicate IDs found:")
    get_duplicate_rows(employees, "id").display()

# Check 2: Composite uniqueness - name + department
print("\n" + "="*60)
print("CHECK 2: Composite Uniqueness (name + department)")
print("="*60)
composite_result = check_uniqueness_composite(employees, ["name", "department"])
print(f"Total Rows: {composite_result['total_rows']}")
print(f"Distinct Combinations: {composite_result['distinct_count']}")
print(f"Duplicate Count: {composite_result['duplicate_count']}")
print(f"Passed: {'✅' if composite_result['passed'] else '❌'}")

# Save results from Scenario 1
scenario1_results = [id_result, composite_result]
scenario1_df = spark.createDataFrame(scenario1_results)
print("\n📊 Scenario 1 Results:")
scenario1_df.display()

In [0]:
"""Scenario 2: Hard-Coded Contact Data
Composite uniqueness check: first_name + last_name + phone

Business Rule: The combination of first_name, last_name, and phone should be unique.
- Individual columns CAN have duplicates (multiple Johns, multiple 555-1234)
- But the COMBINATION of all three together must be unique
"""

from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define schema
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("email", StringType(), True)
])

# Create test data with intentional duplicates
test_data = [
    (1, "John", "Smith", "555-1234", "john.smith1@example.com"),      # First occurrence - UNIQUE
    (2, "John", "Smith", "555-1234", "john.smith2@example.com"),      # ❌ DUPLICATE (same name + phone)
    (3, "John", "Smith", "555-5678", "john.smith3@example.com"),      # ✅ UNIQUE (different phone)
    (4, "Jane", "Doe", "555-1234", "jane.doe@example.com"),           # ✅ UNIQUE (different person)
    (5, "Jane", "Doe", "555-9999", "jane.doe2@example.com"),          # First occurrence - UNIQUE
    (6, "Michael", "Johnson", "555-1111", "michael.j@example.com"),   # ✅ UNIQUE
    (7, "Jane", "Doe", "555-9999", "jane.doe.alt@example.com"),       # ❌ DUPLICATE (same as row 5)
]

# Create DataFrame
contacts_df = spark.createDataFrame(test_data, schema)

print("📊 Contact Test Data:")
contacts_df.display()

print("\n⚠️ Expected Issues:")
print("- Row 2: Duplicate of row 1 (John Smith + 555-1234)")
print("- Row 7: Duplicate of row 5 (Jane Doe + 555-9999)")

# Run composite uniqueness check: first_name + last_name + phone
print("\n" + "="*60)
print("CHECK: Composite Uniqueness (first_name + last_name + phone)")
print("="*60)
composite_check = check_uniqueness_composite(contacts_df, ["first_name", "last_name", "phone"])
print(f"Total Rows: {composite_check['total_rows']}")
print(f"Distinct Combinations: {composite_check['distinct_count']}")
print(f"Duplicate Count: {composite_check['duplicate_count']}")
print(f"Passed: {'✅' if composite_check['passed'] else '❌'}")

# Save results from Scenario 2
scenario2_results = [composite_check]
scenario2_df = spark.createDataFrame(scenario2_results)
print("\n📊 Scenario 2 Results:")
scenario2_df.display()

In [0]:
# Combine results from both scenarios
all_results = scenario1_results + scenario2_results
final_df = spark.createDataFrame(all_results)

print("📊 Final Combined Uniqueness Report:")
final_df.display()

# Save to Delta table
final_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.uniqueness_report")
print("\n✅ Results saved to workspace.default.uniqueness_report")

In [0]:
# Read back the uniqueness report from Delta table
df = spark.table("workspace.default.uniqueness_report")
df.display()